# Demo 06 — Classification & Ensembles

**Machine Learning with scikit-learn — Session 06**

รันตามสไลด์คาบ 6 — เปิด black box ที่เหลือทั้งหมด (logistic, tree, NB, SVM) แล้วปล่อย "คณะกรรมการโมเดล" ลงไปโหวต

## Setup — breast cancer + stratified CV (โปรโตคอลคาบ 4)

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, export_text

X, y = load_breast_cancer(return_X_y=True)
names = load_breast_cancer().feature_names
skf = StratifiedKFold(5, shuffle=True, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

## Slide 4-5 — Logistic regression: a probability pump

z = w·x + b (เหมือน linear) แล้วบีบด้วย sigmoid ให้กลายเป็นความน่าจะเป็น 0-1

In [2]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
print("proba of first 3 patients (benign, malignant):")
print(lr.predict_proba(Xte[:3]).round(3))

proba of first 3 patients (benign, malignant):
[[0.031 0.969]
 [1.    0.   ]
 [0.441 0.559]]


In [3]:
import pandas as pd
coefs = pd.Series(lr[-1].coef_[0], index=names)
coefs.reindex(coefs.abs().sort_values(ascending=False).index)[:5].round(2)

worst texture          -1.25
radius error           -1.07
worst symmetry         -0.96
worst concave points   -0.94
area error             -0.94
dtype: float64

weight คือ "น้ำหนักความเชื่อ" ในหน่วย log-odds (ฟีเจอร์ถูก scale แล้ว) — อ่านขนาดเทียบกันได้ว่าหลักฐานไหนหนักกว่า

## Slide 7 — A real fitted tree, read aloud

ต้นไม้ depth 3 ที่ fit แล้ว — **ตัวโมเดลทั้งชิ้นอ่านเป็นประโยคได้เลย**

In [4]:
tree3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
print(export_text(tree3, feature_names=list(names), max_depth=2))

|--- worst radius <= 16.80
|   |--- worst concave points <= 0.14
|   |   |--- area error <= 91.56
|   |   |   |--- class: 1
|   |   |--- area error >  91.56
|   |   |   |--- class: 0
|   |--- worst concave points >  0.14
|   |   |--- worst texture <= 25.62
|   |   |   |--- class: 1
|   |   |--- worst texture >  25.62
|   |   |   |--- class: 0
|--- worst radius >  16.80
|   |--- texture error <= 0.47
|   |   |--- class: 1
|   |--- texture error >  0.47
|   |   |--- worst concavity <= 0.19
|   |   |   |--- class: 1
|   |   |--- worst concavity >  0.19
|   |   |   |--- class: 0



"ถ้ารัศมีก้อนเนื้อไม่เกิน 16.8 มม. และ concave points ไม่เกิน 0.14 → แนวโน้มเป็นเนื้องอกดี"
 — นี่คือเหตุผลที่โรงพยาบาลชอบ decision tree: audit ทุกการตัดสินใจได้

## Slide 8 — Unleashed, a tree memorizes everything

tree อิสระ vs จำกัด depth — ความจำเป็นพิษ (train 1.000 แต่ CV แย่สุดของวัน)

In [5]:
for name, m in [("free tree", DecisionTreeClassifier(random_state=42)),
                ("depth-3 tree", DecisionTreeClassifier(max_depth=3, random_state=42))]:
    acc = cross_val_score(m, X, y, cv=skf).mean()
    auc = cross_val_score(m, X, y, cv=skf, scoring="roc_auc").mean()
    print(f"{name:12s} CV acc={acc:.3f}  AUC={auc:.3f}")

free tree    CV acc=0.910  AUC=0.900
depth-3 tree CV acc=0.924  AUC=0.934


## Slide 9 — The 90-second tour: Naive Bayes & SVM

In [6]:
for name, m in [("GaussianNB", make_pipeline(StandardScaler(), GaussianNB())),
                ("SVM (RBF)",  make_pipeline(StandardScaler(), SVC(random_state=42))),
                ("KNN (5)",    make_pipeline(StandardScaler(), KNeighborsClassifier()))]:
    acc = cross_val_score(m, X, y, cv=skf).mean()
    auc = cross_val_score(m, X, y, cv=skf, scoring="roc_auc").mean()
    print(f"{name:12s} CV acc={acc:.3f}  AUC={auc:.3f}")

GaussianNB   CV acc=0.930  AUC=0.986


SVM (RBF)    CV acc=0.977  AUC=0.995


KNN (5)      CV acc=0.963  AUC=0.985


## Slide 12 — RandomForest: bagging + a random twist

tree ลึก 100 ต้น แต่ละต้นเห็นข้อมูล/ฟีเจอร์คนละส่วน → โหวต → variance หายไป

In [7]:
rf = RandomForestClassifier(random_state=42)
acc = cross_val_score(rf, X, y, cv=skf).mean()
auc = cross_val_score(rf, X, y, cv=skf, scoring="roc_auc").mean()
print(f"RandomForest(100)  CV acc={acc:.3f}  AUC={auc:.3f}")

RandomForest(100)  CV acc=0.956  AUC=0.989


In [8]:
rf_fit = RandomForestClassifier(random_state=42).fit(Xtr, ytr)
imp = pd.Series(rf_fit.feature_importances_, index=names)
imp.sort_values(ascending=False)[:5].round(3)

worst area              0.150
worst concave points    0.127
mean concave points     0.105
worst radius            0.087
worst perimeter         0.080
dtype: float64

importance บอกว่า "ป่า" ใช้คอลัมน์ไหนจริง ๆ — แต่ระวัง: ฟีเจอร์ที่สัมพันธ์กันเองจะแบ่งเครดิตกัน อ่านเป็นแนวทาง ไม่ใช่คำพิพากษา

## Slide 13 — Boosting: each tree fixes the last one's mistakes

HistGradientBoosting = sklearn's fast boosting (รองรับ NaN ในตัว!) — พี่น้องกับ XGBoost/LightGBM

In [9]:
hgb = HistGradientBoostingClassifier(random_state=42)
acc = cross_val_score(hgb, X, y, cv=skf).mean()
auc = cross_val_score(hgb, X, y, cv=skf, scoring="roc_auc").mean()
print(f"HistGradientBoosting  CV acc={acc:.3f}  AUC={auc:.3f}")

HistGradientBoosting  CV acc=0.967  AUC=0.992


## Slide 14 — Voting: mix different species

สมาชิกควร "ไม่เห็นด้วยกัน" บ้าง (diversity) — logistic + forest + KNN

In [10]:
vote = VotingClassifier([
    ("lr",  make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))),
    ("rf",  RandomForestClassifier(random_state=42)),
    ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier())),
], voting="soft")   # ใช้ probability เฉลี่ย (ดีกว่าโหวตแบบ label)

acc = cross_val_score(vote, X, y, cv=skf).mean()
auc = cross_val_score(vote, X, y, cv=skf, scoring="roc_auc").mean()
print(f"Voting(LR+RF+KNN)  CV acc={acc:.3f}  AUC={auc:.3f}")

Voting(LR+RF+KNN)  CV acc=0.972  AUC=0.995


สรุปสไลด์ 15: บนข้อมูลง่ายแบบนี้ SVM (0.977) ชนะเฉียด ๆ — **ไม่มีโมเดลชนะทุกที่** วิธีเทียบของคาบ 4 ต่างหากคือทักษะที่แท้จริง

---
✅ **จบ Demo 06** — **Lab 06**: `excercises/06.classification_ensembles_exercise.ipynb`